<a href="https://colab.research.google.com/github/lzxatdk-tech/NLP_project/blob/main/reliable_multilingual_question_answering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment Setup

In [18]:
import random
import re

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

random.seed(42)
np.random.seed(42)

## Mount data folder

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
folder_path = '/content/drive/MyDrive/Shared_NLP_Project'

os.chdir(folder_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Download or load dataset

In [2]:
import os
from datasets import load_dataset, load_from_disk

TARGET_DIR = "./tydi_xor_rc"

if not os.path.exists(TARGET_DIR):
    print(f"Downloading dataset to {TARGET_DIR}...")
    dataset = load_dataset("coastalcph/tydi_xor_rc")
    dataset.save_to_disk(TARGET_DIR)
    print("Download and save complete.")
else:
    print(f"Found existing dataset at {TARGET_DIR}. Loading from disk...")
    dataset = load_from_disk(TARGET_DIR)
    print("Loading completes.")

df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

df_train = df_train[
    df_train["lang"].isin(["ar", "ko", "te"])
]

df_val = df_val[
    df_val["lang"].isin(["ar", "ko", "te"])
]

Found existing dataset at ./tydi_xor_rc. Loading from disk...
Loading completes.


In [3]:
# tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

In [4]:
tokenizer.tokenize("The exact number of Arab casualties is unknown. One estimate places the Arab death toll at 7,000,")

['▁The',
 '▁exact',
 '▁number',
 '▁of',
 '▁Arab',
 '▁casual',
 'ties',
 '▁is',
 '▁un',
 'know',
 'n',
 '.',
 '▁One',
 '▁estima',
 'te',
 '▁places',
 '▁the',
 '▁Arab',
 '▁death',
 '▁toll',
 '▁at',
 '▁',
 '7,000',
 ',']

In [5]:
%pip install --upgrade deepl

In [6]:
import deepl

translator = deepl.Translator("d4c7a259-bfa3-4859-ae03-4e17036caa51:fx")

TE = "TE"
AR = "AR"
KO = "KO"
EN = "EN-US"

def translate(sentence, lang):
  if lang == "ko":
    source_lang = KO
  elif lang == "te":
    source_lang = TE
  else:
    source_lang = AR

  return translator.translate_text(sentence, source_lang=source_lang, target_lang=EN).text

## BIO Labels

In [7]:
def is_answer_valid(answer, answer_start, context):
  return not (not answer or answer_start is None or answer_start < 0)

def bio_labelller(context, answer, answer_start):
  encoding = tokenizer(
      context,
      add_special_tokens=False,
      return_offsets_mapping=True,
  )

  tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
  offsets = encoding["offset_mapping"]

  labels = ["O"] * len(tokens)

  if not is_answer_valid(answer, answer_start, context):
      return tokens, offsets, labels

  answer_end = answer_start + len(answer)

  for token_index, (token_start, token_end) in enumerate(offsets):
      if token_start <= answer_start and token_end > answer_start:
          labels[token_index] = "B"
      elif token_start < answer_end and token_start > answer_start:
          labels[token_index] = "I"

  return tokens, offsets, labels

def bio_labelller_row(row):
  context = row["context"]
  answer = row["answer"]
  answer_start = row["answer_start"]

  tokens, offsets, labels = bio_labelller(context, answer, answer_start)

  return labels

def tokenizer_row(row):
  context = row["context"]
  answer = row["answer"]
  answer_start = row["answer_start"]

  tokens, offsets, labels = bio_labelller(context, answer, answer_start)

  return tokens


In [8]:
df_train["sequence_labels"] = df_train.apply(
    bio_labelller_row,
    axis=1,
)

df_train["context_tokens"] = df_train.apply(
    tokenizer_row,
    axis=1,
)

df_val["sequence_labels"] = df_val.apply(
    bio_labelller_row,
    axis=1,
)

df_val["context_tokens"] = df_val.apply(
    tokenizer_row,
    axis=1,
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (547 > 512). Running this sequence through the model will result in indexing errors


In [9]:
df_train.head()

,question,context,lang,answerable,answer_start,answer,answer_inlang,sequence_labels,context_tokens
4792,30년 전쟁의 승자는 누구인가?,The conflict between France and Spain continue...,ko,True,21,France,None,"[O, O, O, B, O, O, O, O, O, O, O, O, O, O, O, ...","[▁The, ▁conflict, ▁between, ▁France, ▁and, ▁Sp..."
4793,엑스선은 누가 발견하였는가?,"X-rays make up X-radiation, a form of electrom...",ko,True,503,Wilhelm Röntgen,None,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[▁X, -, ray, s, ▁make, ▁up, ▁X, -, radi, ation..."
4794,아테네에서 언제 가장 최근의 올림픽이 올렸나요?,"In 2022, Beijing will become the first-ever ci...",ko,True,188,2004,None,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[▁In, ▁2022, ,, ▁Beijing, ▁will, ▁become, ▁the..."
4795,세상에서 가장 오래된 방송사는 무엇인가?,The British Broadcasting Corporation (BBC) is ...,ko,True,4,British Broadcasting Corporation (BBC),None,"[O, B, I, I, I, I, I, I, O, O, O, O, O, O, O, ...","[▁The, ▁British, ▁Broadcast, ing, ▁Corporation..."
4796,팔레스타인 수도는 어딘가요?,"Palestine ( '), officially the State of Palest...",ko,True,205,Jerusalem,None,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[▁Palestin, e, ▁(, ▁', ),, ▁official, ly, ▁the..."


## Automatic checks

In [10]:
def answers_at_0():
  return df_train[
      df_train["answer_start"] == 0
  ]

def multi_token_answers():
  return df_train[
    df_train["sequence_labels"].apply(
        lambda labels: sum(
            label in {"B", "I"} for label in labels
        ) > 1
    )
]

import unicodedata

def is_punctuation(character):
  return (
      character is not None
      and unicodedata.category(character).startswith("P")
  )

def has_adjacent_punctuation(row):
  context = row["context"]
  answer = row["answer"]
  answer_start = row["answer_start"]

  answer_start = int(answer_start)

  if answer_start < 0:
      return False

  answer_end = answer_start + len(answer)

  character_before = (
      context[answer_start - 1]
      if answer_start > 0
      else None
  )

  character_after = (
      context[answer_end]
      if answer_end < len(context)
      else None
  )

  return (
      is_punctuation(character_before)
      or is_punctuation(character_after)
  )

def answers_adjacent_to_punctuation():
  return df_train[
      df_train.apply(
          has_adjacent_punctuation,
          axis=1,
      )
  ]

def unanswerable_answers():
  return df_train[
      df_train["answerable"] == False
  ]


In [11]:
def test_answer_at_0():
  rows = answers_at_0().sample(5)

  for row in rows.itertuples():
    sequence_labels = row.sequence_labels

    assert sequence_labels[0] == "B"

  print("All tests on answers at 0 passed.")

test_answer_at_0()

All tests on answers at 0 passed.


In [12]:
def test_multi_token_answers():
  rows = multi_token_answers().sample(5)

  for row in rows.itertuples():
    sequence_labels = row.sequence_labels

    assert sequence_labels.count("B") + sequence_labels.count("I") > 1

  print("All tests on multi-token answers passed.")

test_multi_token_answers()

All tests on multi-token answers passed.


In [13]:
def test_answers_adjacent_to_punctuation():
  rows = answers_adjacent_to_punctuation().sample(5)

  for row in rows.itertuples():
    answer = row.answer
    answer_start = row.answer_start
    answer_end = answer_start + len(answer)
    context = row.context

    char_before = None
    char_after = None
    if answer_start > 0:
      char_before = context[answer_start - 1]

    if answer_end < len(context):
      char_after = context[answer_end]

    assert (
        is_punctuation(char_before)
        or is_punctuation(char_after)
    )
  print("All tests on answers adjacent to punctuations passed.")

test_answers_adjacent_to_punctuation()


All tests on answers adjacent to punctuations passed.


In [14]:
def test_unanswerable_answers():
  rows = unanswerable_answers().sample(5)

  for row in rows.itertuples():
    answer_start = row.answer_start
    sequence_labels = row.sequence_labels

    assert answer_start == -1
    assert sequence_labels.count("B") + sequence_labels.count("I") == 0

  print("All tests on unanswerable answers passed.")

test_unanswerable_answers()

All tests on unanswerable answers passed.


# Question-conditioned sequence labeller

In [42]:
LABELS = ["O", "B", "I"]
LABEL_TO_ID = {"O": 0, "B": 1, "I": 2}

# def token_features(question_tokens, context_tokens, index):
#     token = context_tokens[index]
#     normalized_token = normalize_token(token)

#     question_set = {
#         normalize_token(question_token)
#         for question_token in question_tokens
#     }

#     question_words = question_set & QUESTION_WORDS

#     features = {
#         "bias": 1.0,
#         "token.lower": normalized_token,
#         "token.prefix2": normalized_token[:2],
#         "token.suffix2": normalized_token[-2:],
#         "token.isdigit": normalized_token.isdigit(),
#         "token.istitle": normalized_token.istitle(),
#         "token_in_question": normalized_token in question_set,
#         "relative_position": index / max(len(context_tokens), 1),
#     }

#     if index > 0:
#         previous = normalize_token(context_tokens[index - 1])

#         features["previous.lower"] = previous
#         features["previous_in_question"] = (
#             previous in question_set
#         )

#     if index + 1 < len(context_tokens):
#         following = normalize_token(context_tokens[index + 1])

#         features["next.lower"] = following
#         features["next_in_question"] = (
#             following in question_set
#         )

#     for question_word in question_words:
#         features[f"question_word={question_word}"] = 1.0

#         # Question-word and context-token-property interactions
#         features[
#             f"question_word={question_word}|isdigit={normalized_token.isdigit()}"
#         ] = 1.0

#         features[
#             f"question_word={question_word}|istitle={normalized_token.istitle()}"
#         ] = 1.0

#     return features


def token_features(question_tokens, context_tokens, index):
    token = context_tokens[index]
    question_set = {
        question_token.lower()
        for question_token in question_tokens
    }

    features = {
        "bias": 1.0,
        "token.lower": token.lower(),
        "token.prefix2": token[:2].lower(),
        "token.suffix2": token[-2:].lower(),
        "token.isdigit": token.isdigit(),
        "token_in_question": token.lower() in question_set,
        "relative_position": index / max(len(context_tokens), 1),
    }

    if index > 0:
        features["previous.lower"] = context_tokens[index - 1].lower()

    if index + 1 < len(context_tokens):
        features["next.lower"] = context_tokens[index + 1].lower()

    # Add the question representation to every context token
    for question_token in question_set:
        features[f"question_token={question_token}"] = 1.0
        features[
            f"question_context_pair={question_token}|{token.lower()}"
        ] = 1.0

    return features

def featurize_split(data):
    features, labels, lengths = [], [], []

    for row in data.itertuples():
      question = row.question
      context = row.context
      sequence_labels = row.sequence_labels
      tokens = row.context_tokens

      question_tokens = tokenizer.tokenize(translate(question, row.lang))
      lengths.append(len(tokens))
      features.extend(token_features(question_tokens, tokens, index) for index in range(len(tokens)))
      labels.extend(sequence_labels)

    return features, labels, lengths

def split_by_lengths(values, lengths):
    sequences = []
    offset = 0
    for length in lengths:
        sequences.append(list(values[offset:offset + length]))
        offset += length
    assert offset == len(values)
    return sequences

In [43]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

vectorizer = DictVectorizer(sparse=True)

train_features, train_labels, train_lengths = featurize_split(df_train)

X_train = vectorizer.fit_transform(train_features)

token_classifier = SGDClassifier(
    loss="log_loss",
    alpha=1e-5,
    class_weight="balanced",
    max_iter=30,
    random_state=42,
)
token_classifier.fit(X_train, train_labels)

SGDClassifier(alpha=1e-05, class_weight='balanced', loss='log_loss',
              max_iter=30, random_state=42)

In [41]:
validation_features, validation_labels, validation_lengths = featurize_split(df_val.head(200))

X_validation = vectorizer.transform(validation_features)

independent_flat = token_classifier.predict(X_validation)
independent_predictions = split_by_lengths(
    independent_flat, validation_lengths
)

In [39]:
def bio_spans(labels):
    spans = set()
    start = None

    for index, label in enumerate(list(labels) + ["O"]):
      if label == "B":
        if start is not None:
            spans.add((start, index))
        start = index

      elif label == "I":
        if start is None:
            start = index

      elif label == "O" and start is not None:
        spans.add((start, index))
        start = None

    return spans


def span_scores(gold_sequences, predicted_sequences):
    predicted_count = gold_count = matches = 0
    for gold, predicted in zip(gold_sequences, predicted_sequences):
        gold_spans = bio_spans(gold)
        predicted_spans = bio_spans(predicted)
        gold_count += len(gold_spans)
        predicted_count += len(predicted_spans)
        matches += len(gold_spans & predicted_spans)

    precision = matches / predicted_count if predicted_count else 0.0
    recall = matches / gold_count if gold_count else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


# def evaluate_sequences(name, gold_sequences, predicted_sequences):
#     gold_flat = np.concatenate(gold_sequences)
#     predicted_flat = np.concatenate(predicted_sequences)
#     precision, recall, span_f1 = span_scores(gold_sequences, predicted_sequences)
#     result = {
#         "model": name,
#         "token_accuracy": accuracy_score(gold_flat, predicted_flat),
#         "entity_token_macro_f1": f1_score(
#             gold_flat,
#             predicted_flat,
#             labels=list(range(1, len(LABELS))),
#             average="macro",
#             zero_division=0,
#         ),
#         "span_precision": precision,
#         "span_recall": recall,
#         "span_f1": span_f1,
#     }
#     return result

def evaluate_sequences(name, gold_sequences, predicted_sequences):
    gold_flat = np.concatenate(gold_sequences)
    predicted_flat = np.concatenate(predicted_sequences)

    # B and I are both positive; O is negative.
    gold_answer_tokens = np.isin(gold_flat, ["B", "I"])
    predicted_answer_tokens = np.isin(
        predicted_flat, ["B", "I"]
    )

    exact_matches = [
        bio_spans(gold) == bio_spans(predicted)
        for gold, predicted in zip(
            gold_sequences,
            predicted_sequences,
        )
    ]

    unanswerable = [
        len(bio_spans(gold)) == 0
        for gold in gold_sequences
    ]

    return {
        "model": name,
        "token_accuracy": accuracy_score(
            gold_flat,
            predicted_flat,
        ),
        "answer_token_f1": f1_score(
            gold_answer_tokens,
            predicted_answer_tokens,
            zero_division=0,
        ),
        "exact_span_match": np.mean(exact_matches),
        "unanswerable_exact_match": np.mean([
            exact
            for exact, is_unanswerable in zip(
                exact_matches,
                unanswerable,
            )
            if is_unanswerable
        ]),
    }


In [40]:
validation_gold = split_by_lengths(
    validation_labels, validation_lengths
)

always_o_predictions = [
    ["O"] * length
    for length in validation_lengths
]

print(evaluate_sequences(
    "Always O",
    validation_gold,
    always_o_predictions,
))

print(evaluate_sequences(
    "Question-conditioned classifier",
    validation_gold,
    independent_predictions,
))

{'model': 'Always O', 'token_accuracy': 0.9734098018769551, 'answer_token_f1': 0.0, 'exact_span_match': np.float64(0.17), 'unanswerable_exact_match': np.float64(1.0)}
{'model': 'Question-conditioned classifier', 'token_accuracy': 0.9723670490093848, 'answer_token_f1': 0.061946902654867256, 'exact_span_match': np.float64(0.15), 'unanswerable_exact_match': np.float64(0.8823529411764706)}
